# Amazon Search Ranking — Week 8 V3 Error Analysis

**Upload only 3 ZIP files:**
1. `amazon_week5_v2_lambdamart_results.zip`
2. `amazon_week5_v3_model_comparison_ablation_results.zip`
3. `amazon_week4_v2_clip_multimodal_results.zip`

The notebook produces query-level reranking diagnostics, representative failures, full-catalog retrieval failures, charts, and report-ready analysis.

In [ ]:

# ============================================================
# AMAZON SEARCH RANKING — WEEK 8 V3 ERROR ANALYSIS (ONE CLICK)
#
# Upload ONLY these 3 files:
# 1) amazon_week5_v2_lambdamart_results.zip
# 2) amazon_week5_v3_model_comparison_ablation_results.zip
# 3) amazon_week4_v2_clip_multimodal_results.zip
#
# Output:
# amazon_week8_v3_error_analysis_results.zip
# ============================================================

import os, zipfile, shutil, glob, re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score

try:
    from google.colab import files
    uploaded = files.upload()
    paths = [os.path.abspath(n) for n in uploaded.keys()]
except Exception:
    paths = glob.glob("*.zip")

def pick(pattern):
    hits = [p for p in paths if pattern.lower() in os.path.basename(p).lower()]
    if not hits:
        raise FileNotFoundError(f"Missing required ZIP: {pattern}")
    return hits[0]

W5V2 = pick("week5_v2_lambdamart_results")
W5V3 = pick("week5_v3_model_comparison_ablation_results")
W4V2 = pick("week4_v2_clip_multimodal_results")

OUT = Path("/content/week8_v3_error_analysis_results")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

def read_csv_zip(zp, member):
    with zipfile.ZipFile(zp) as z:
        with z.open(member) as f:
            return pd.read_csv(f)

rankings = read_csv_zip(W5V2, "week5_v2_test_rankings.csv")
qcomp = read_csv_zip(W5V3, "week5_v3_querywise_xgb_vs_lambdamart.csv")
retq = read_csv_zip(W4V2, "week4_v2_per_query_analysis.csv")
fusion_top50 = read_csv_zip(W4V2, "week4_multimodal_top50.csv")

GENERIC_WORDS = {
    "great","good","amazing","excellent","nice","best","wonderful","lovely",
    "quality","soft","smooth","strong","terrible","bad","perfect","love",
    "favorite","awesome","realistic","natural","product","find","beautiful",
    "works","well","cute","pretty","prep"
}
REVIEW_WORDS = {"i","my","me","this","again","purchase","favorite","product"}

def query_characteristic(q):
    toks = re.findall(r"[a-z0-9]+", str(q).lower())
    generic_count = sum(t in GENERIC_WORDS for t in toks)
    review_count = sum(t in REVIEW_WORDS for t in toks)
    if (len(toks) <= 5 and generic_count >= 1) or generic_count >= 2 or review_count >= 2:
        return "Generic / subjective / review-like"
    if len(toks) >= 6:
        return "Long specific phrase"
    return "Short product / attribute query"

def signal_ndcg(g, score_col):
    y_true = g["relevance_label"].fillna(0).to_numpy().reshape(1,-1)
    y_score = g[score_col].fillna(0).to_numpy().reshape(1,-1)
    return float(ndcg_score(y_true, y_score, k=10))

def count_inversions(g):
    ordered = g.sort_values("predicted_rank")
    rels = ordered["relevance_label"].fillna(0).to_list()
    inv = 0
    for i in range(len(rels)):
        for j in range(i+1, len(rels)):
            if rels[i] < rels[j]:
                inv += 1
    return inv

rows = []
for qid, g in rankings.groupby("query_id"):
    q = g["query"].iloc[0]
    official = qcomp[qcomp["query_id"].eq(qid)].iloc[0]
    lm = float(official["LambdaMART_NDCG@10"])
    xgb = float(official["XGBoost_NDCG@10"])
    xgb_gap = xgb - lm

    sigs = {
        "BM25": signal_ndcg(g,"bm25_score"),
        "TF-IDF": signal_ndcg(g,"tfidf_similarity"),
        "SBERT": signal_ndcg(g,"sbert_similarity"),
        "CLIP": signal_ndcg(g,"clip_similarity")
    }
    best_signal = max(sigs, key=sigs.get)
    best_signal_ndcg = sigs[best_signal]
    best_gain = best_signal_ndcg - lm

    pred = g.sort_values("predicted_rank")
    top1 = pred.iloc[0]
    ideal = g.sort_values(["relevance_label","lambdamart_score"],ascending=[False,False]).iloc[0]
    max_rel = float(g["relevance_label"].max())
    top1_rel = float(top1["relevance_label"])
    char = query_characteristic(q)

    if best_gain >= 0.15:
        primary = {
            "BM25":"Lexical evidence underweighted",
            "TF-IDF":"Lexical evidence underweighted",
            "SBERT":"Semantic evidence underweighted",
            "CLIP":"Multimodal evidence underweighted"
        }[best_signal]
    elif xgb_gap >= 0.15:
        primary = "LambdaMART-specific ranking instability"
    elif char == "Generic / subjective / review-like" and lm < 0.85:
        primary = "Ambiguous / subjective query"
    elif top1_rel < max_rel:
        primary = "Top-1 ranking inversion"
    else:
        primary = "Cross-signal fine-ordering conflict"

    severity = "Critical" if lm < 0.70 else "Major" if lm < 0.80 else "Moderate" if lm < 0.90 else "Minor"

    if "underweighted" in primary:
        diagnosis = (
            f"{best_signal} alone reaches NDCG@10={best_signal_ndcg:.3f}, "
            f"{best_gain:.3f} above LambdaMART; the combined ranker appears to underweight this signal."
        )
    elif primary == "LambdaMART-specific ranking instability":
        diagnosis = (
            f"XGBoost reaches NDCG@10={xgb:.3f}, beating LambdaMART by {xgb_gap:.3f}; "
            "this suggests model/objective instability rather than missing evidence."
        )
    elif primary == "Ambiguous / subjective query":
        diagnosis = (
            "Subjective or review-like wording provides weak product-identifying evidence, "
            "causing conflicting ranking signals."
        )
    elif primary == "Top-1 ranking inversion":
        diagnosis = (
            f"A relevance-{int(top1_rel)} item is ranked first although relevance-{int(max_rel)} exists."
        )
    else:
        diagnosis = "No single signal clearly explains the remaining fine-ordering error."

    rows.append({
        "query_id":qid,"query":q,"severity":severity,
        "query_characteristic":char,"primary_error_category":primary,
        "LambdaMART_NDCG@10":lm,"XGBoost_NDCG@10":xgb,
        "XGBoost_minus_LambdaMART":xgb_gap,
        "BM25_NDCG@10":sigs["BM25"],"TFIDF_NDCG@10":sigs["TF-IDF"],
        "SBERT_NDCG@10":sigs["SBERT"],"CLIP_NDCG@10":sigs["CLIP"],
        "best_base_signal":best_signal,
        "best_base_signal_NDCG@10":best_signal_ndcg,
        "best_signal_gain_over_LambdaMART":best_gain,
        "top1_product_id":top1["product_id"],"top1_title":top1["title"],
        "top1_relevance":top1_rel,
        "ideal_product_id":ideal["product_id"],"ideal_title":ideal["title"],
        "max_relevance":max_rel,"pairwise_inversions":count_inversions(g),
        "diagnosis":diagnosis
    })

diag = pd.DataFrame(rows).sort_values(
    ["LambdaMART_NDCG@10","pairwise_inversions"],ascending=[True,False]
).reset_index(drop=True)
diag.to_csv(OUT/"week8_v3_all_query_error_diagnostics.csv",index=False)

# representative reranking failures
selected = []
cats = [
    "Lexical evidence underweighted","Semantic evidence underweighted",
    "Multimodal evidence underweighted","LambdaMART-specific ranking instability",
    "Ambiguous / subjective query","Top-1 ranking inversion",
    "Cross-signal fine-ordering conflict"
]
for cat in cats:
    sub = diag[(diag["primary_error_category"]==cat) & (diag["LambdaMART_NDCG@10"]<0.90)]
    for idx in sub.head(4).index:
        if idx not in selected:
            selected.append(idx)
for idx in diag.index:
    if len(selected) >= 25:
        break
    if idx not in selected and diag.loc[idx,"LambdaMART_NDCG@10"] < 0.90:
        selected.append(idx)

rep = diag.loc[selected].sort_values("LambdaMART_NDCG@10").reset_index(drop=True)
rep.insert(0,"case_id",[f"E{i:02d}" for i in range(1,len(rep)+1)])
rep.to_csv(OUT/"week8_v3_representative_reranking_failures.csv",index=False)

failure_pool = diag[diag["LambdaMART_NDCG@10"]<0.90]
cat_summary = failure_pool.groupby("primary_error_category").agg(
    query_count=("query_id","count"),
    mean_LambdaMART_NDCG10=("LambdaMART_NDCG@10","mean"),
    mean_XGB_minus_Lambda=("XGBoost_minus_LambdaMART","mean"),
    mean_pairwise_inversions=("pairwise_inversions","mean")
).reset_index()
cat_summary["share_of_failures_pct"] = cat_summary["query_count"]/len(failure_pool)*100
cat_summary = cat_summary.sort_values("query_count",ascending=False)
cat_summary.to_csv(OUT/"week8_v3_error_category_summary.csv",index=False)

diag.groupby("severity").agg(
    query_count=("query_id","count"),
    mean_NDCG10=("LambdaMART_NDCG@10","mean")
).reset_index().to_csv(OUT/"week8_v3_error_severity_summary.csv",index=False)

# retrieval errors
expected = rankings.sort_values(
    ["query_id","relevance_label","lambdamart_score"],ascending=[True,False,False]
).groupby("query_id",as_index=False).first()[
    ["query_id","product_id","title","relevance_label"]
].rename(columns={
    "product_id":"expected_product_id","title":"expected_best_title",
    "relevance_label":"expected_max_relevance"
})

fusion_top1 = fusion_top50.sort_values(["query_id","rank"]).groupby(
    "query_id",as_index=False
).first()[["query_id","product_id","title","relevance_label","rank"]].rename(columns={
    "product_id":"fusion_top1_product_id","title":"fusion_top1_title",
    "relevance_label":"fusion_top1_relevance","rank":"fusion_top1_rank"
})

tmp = fusion_top50.merge(expected[["query_id","expected_product_id"]],on="query_id",how="left")
tmp["is_expected"] = tmp["product_id"] == tmp["expected_product_id"]
expected_rank = tmp[tmp["is_expected"]].groupby("query_id")["rank"].min().rename(
    "expected_product_fusion_rank"
).reset_index()

retr = retq.merge(expected,on="query_id",how="left").merge(
    fusion_top1,on="query_id",how="left"
).merge(expected_rank,on="query_id",how="left")

def retrieval_category(r):
    if r["Recall@50_full_FUSION"] < r["Recall@50_full_SBERT"] - 0.49:
        return "Fusion coverage regression"
    if r["NDCG@10_judged_FUSION"] < r["NDCG@10_judged_SBERT"] - 0.15:
        return "Fusion ranking regression"
    if r["Recall@50_full_FUSION"] == 0 and r["NDCG@10_judged_FUSION"] < 0.65:
        return "Coverage failure + weak ordering"
    if r["Recall@50_full_FUSION"] == 0:
        return "Coverage failure"
    if r["NDCG@10_judged_FUSION"] < 0.65:
        return "Weak top-10 ordering"
    return "Minor retrieval miss"

retr["retrieval_error_category"] = retr.apply(retrieval_category,axis=1)
retr["retrieval_failure_score"] = (
    (1-retr["NDCG@10_judged_FUSION"]) +
    (1-retr["Recall@50_full_FUSION"])*0.6 +
    np.maximum(0,-retr["NDCG_fusion_minus_SBERT"])*0.5
)
retr["expected_found_in_fusion_top50"] = retr["expected_product_fusion_rank"].notna()

cols = [
    "query_id","query","retrieval_error_category","retrieval_failure_score",
    "NDCG@10_judged_FUSION","Recall@50_full_FUSION",
    "NDCG@10_judged_SBERT","Recall@50_full_SBERT",
    "NDCG_fusion_minus_SBERT","Recall50_fusion_minus_SBERT",
    "expected_product_id","expected_best_title","expected_max_relevance",
    "fusion_top1_product_id","fusion_top1_title","fusion_top1_relevance",
    "expected_found_in_fusion_top50","expected_product_fusion_rank"
]
retr_all = retr[cols].sort_values("retrieval_failure_score",ascending=False)
retr_all.to_csv(OUT/"week8_v3_all_retrieval_error_diagnostics.csv",index=False)

retr_rep = retr_all.head(15).copy()
retr_rep.insert(0,"case_id",[f"R{i:02d}" for i in range(1,len(retr_rep)+1)])
retr_rep.to_csv(OUT/"week8_v3_representative_retrieval_failures.csv",index=False)

retr_summary = retr_all.groupby("retrieval_error_category").agg(
    query_count=("query_id","count"),
    mean_fusion_NDCG10=("NDCG@10_judged_FUSION","mean"),
    mean_fusion_Recall50=("Recall@50_full_FUSION","mean"),
    mean_delta_vs_SBERT=("NDCG_fusion_minus_SBERT","mean")
).reset_index().sort_values("query_count",ascending=False)
retr_summary["share_pct"] = retr_summary["query_count"]/len(retr_all)*100
retr_summary.to_csv(OUT/"week8_v3_retrieval_error_summary.csv",index=False)

# plots
plt.figure(figsize=(8,5))
plt.hist(diag["LambdaMART_NDCG@10"],bins=15)
plt.xlabel("Per-query LambdaMART NDCG@10")
plt.ylabel("Number of queries")
plt.title("Week 8 V3 — Distribution of Reranking Quality")
plt.tight_layout()
plt.savefig(OUT/"week8_v3_reranking_ndcg_distribution.png",dpi=180)
plt.close()

plt.figure(figsize=(9,5))
p = cat_summary.sort_values("query_count",ascending=True)
plt.barh(p["primary_error_category"],p["query_count"])
plt.xlabel("Queries with NDCG@10 < 0.90")
plt.title("Week 8 V3 — Reranking Error Categories")
plt.tight_layout()
plt.savefig(OUT/"week8_v3_error_category_counts.png",dpi=180)
plt.close()

plt.figure(figsize=(10,6))
w = diag.head(15).sort_values("LambdaMART_NDCG@10",ascending=True)
labels = [f"{qid}: {q[:35]}" for qid,q in zip(w["query_id"],w["query"])]
plt.barh(labels,w["LambdaMART_NDCG@10"])
plt.xlabel("LambdaMART NDCG@10")
plt.title("Week 8 V3 — 15 Worst Reranking Queries")
plt.tight_layout()
plt.savefig(OUT/"week8_v3_worst15_reranking_queries.png",dpi=180)
plt.close()

plt.figure(figsize=(8,6))
plt.scatter(retq["Recall@50_full_FUSION"],retq["NDCG@10_judged_FUSION"])
plt.xlabel("Fusion Recall@50")
plt.ylabel("Fusion NDCG@10")
plt.title("Week 8 V3 — Full-Catalog Retrieval Failure Map")
plt.tight_layout()
plt.savefig(OUT/"week8_v3_retrieval_failure_map.png",dpi=180)
plt.close()

# report-ready text
n_fail = len(failure_pool)
n_critical = int((diag["severity"]=="Critical").sum())
n_major = int((diag["severity"]=="Major").sum())
worst = diag.iloc[0]
most_common = cat_summary.iloc[0]
fusion_regress = int((retr["NDCG_fusion_minus_SBERT"] < -0.15).sum())
coverage_regress = int((retr["Recall50_fusion_minus_SBERT"] < 0).sum())

example_lines=[]
for _,r in rep.head(8).iterrows():
    example_lines.append(
        f"- **{r['query']}** — NDCG@10={r['LambdaMART_NDCG@10']:.3f}; "
        f"{r['primary_error_category']}. {r['diagnosis']}"
    )

report=f"""# Week 8 V3 — Error Analysis

## Scope
This analysis separates reranking errors from full-catalog retrieval errors.

## Reranking overview
Out of 200 test queries, {n_fail} have LambdaMART NDCG@10 below 0.90.
Critical (<0.70): {n_critical}. Major (0.70–0.80): {n_major}.
Worst query: "{worst['query']}" with NDCG@10={worst['LambdaMART_NDCG@10']:.3f}.

Most common diagnosed category:
{most_common['primary_error_category']} — {int(most_common['query_count'])} queries
({most_common['share_of_failures_pct']:.1f}% of analyzed failures).

## Representative failures
{chr(10).join(example_lines)}

## Retrieval findings
{coverage_regress} queries lose Recall@50 when moving from SBERT to SBERT+CLIP fusion.
{fusion_regress} queries lose more than 0.15 NDCG@10 versus SBERT.

## Interpretation
The remaining errors come from three main sources:
1. subjective/review-like queries with weak identifying information;
2. query-specific conflicts among lexical, semantic, and multimodal signals;
3. retrieval coverage failures that happen before reranking.

## Recommended improvements
- Query-type classifier for product/brand, attribute, and subjective queries.
- Query-adaptive fusion weights.
- Lexical safety path for exact/brand-heavy queries.
- Re-tune CLIP fusion on queries where Recall@50 decreases.
- Add hard negatives from these failure cases to future ranking training.
"""
(OUT/"week8_v3_report_ready_error_analysis.md").write_text(report,encoding="utf-8")

pd.DataFrame([
    {"file":p.name,"purpose":"Week 8 V3 error-analysis output"}
    for p in sorted(OUT.iterdir())
]).to_csv(OUT/"week8_v3_manifest.csv",index=False)

ZIP="/content/amazon_week8_v3_error_analysis_results.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        z.write(p,arcname=p.name)

print("=== WEEK 8 V3 COMPLETE ===")
print(f"Queries below NDCG@10 0.90: {n_fail}/200")
print(f"Critical failures: {n_critical}")
print("\nTop error categories:")
print(cat_summary[["primary_error_category","query_count","share_of_failures_pct"]].to_string(index=False))
print("\nWorst 10 reranking queries:")
print(diag[["query_id","query","LambdaMART_NDCG@10","primary_error_category"]].head(10).to_string(index=False))
print("\nOutput:",ZIP)

try:
    files.download(ZIP)
except Exception:
    pass
